# 綱要 PDF → Markdown（每篇一段＋品質報告）

**不需要 GPU**（CPU 即可，約 1–2 分鐘）。讀 PDF 的文字層（不做 OCR，字元不會辨識錯），去掉頁首頁尾、頁碼與目錄行，
修正 PDF 字型常見的替代字（康熙部首「⼀」、全形數字、經節範圍的「∼」），把換行接回同一個要點，依「第X篇」分篇。

輸出寫在 PDF 旁邊的 `<檔名>-md/` 資料夾（綱要有著作權，只放在你的雲端硬碟，不進 Git）：
- `<檔名>.md`：全部篇；`<檔名>/NN.md`：每篇一個檔，聚會前貼進「本場綱要」用。
- `<檔名>.report.json`：頁數、沒有文字層的頁、每篇的讀經行數／要點數／經節數、警告。

跑完回覆「直接讀」，我用 Google Drive 工具讀報告與 Markdown。

In [ ]:
# 第 1 格：設定
PDF_NAME = 'ex_TS26TRN_zht.pdf'   # 雲端硬碟上的綱要 PDF 檔名（會自動搜尋 MyDrive 前 4 層資料夾）

In [ ]:
# 第 2 格：連接 Google Drive、找到 PDF
import os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive')
found = [Path(d)/PDF_NAME for d, _, files in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) <= 4 and PDF_NAME in files]
if not found:
    raise RuntimeError(f'找不到 {PDF_NAME}；請確認檔名，或把 PDF 放在 MyDrive 前 4 層資料夾內')
PDF = found[0]
OUTPUT = PDF.parent/(PDF.stem + '-md')
print('PDF：', PDF, f'（{PDF.stat().st_size/1e6:.1f} MB）')

In [ ]:
# 第 3 格：取得固定版本的程式與套件
COMMIT = '1903508beb9ded501735ceb93c8832fc8b50be65'
REPO = Path('/content/Meeting-Live-Captions')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/timo893131/Meeting-Live-Captions.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '-q', 'origin', COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pymupdf==1.26.5'], check=True)
print('程式版本', COMMIT[:12])

In [ ]:
# 第 4 格：轉換
result = subprocess.run([sys.executable, str(REPO/'scripts/outline_pdf_to_md.py'), str(PDF), '--output', str(OUTPUT)],
                        capture_output=True, text=True, cwd=REPO)
print(result.stdout[-20000:], result.stderr[-5000:])
if result.returncode:
    raise RuntimeError('轉換失敗，請看上面的錯誤訊息')
print('輸出：', OUTPUT)